In [1]:
import pandas as pd
import os
import glob

# Find the oxygen-saturation export in the repository raw-data directory.
try:
    data_dir = '../../data/raw/samsung_health/export_20260516'
    oxygen_files = glob.glob(os.path.join(data_dir, 'com.samsung.shealth.tracker.oxygen_saturation.*.csv'))
    if not oxygen_files:
        raise FileNotFoundError(f"No oxygen-saturation CSV file found in {data_dir}.")
    INPUT_FILE = oxygen_files[0]  # Use the first match
    print(f"Found input file: {INPUT_FILE}")
except FileNotFoundError as e:
    print(e)
    INPUT_FILE = None

OUTPUT_FILE = "../../data/processed/daily_oxygen_saturation_summary.csv"

START_TIME_COL = "com.samsung.health.oxygen_saturation.start_time"
CREATE_SH_VER_COL = "com.samsung.health.oxygen_saturation.create_sh_ver"
PKG_NAME_COL = "com.samsung.health.oxygen_saturation.pkg_name"
SPO2_COL = "com.samsung.health.oxygen_saturation.spo2"
MAX_COL = "com.samsung.health.oxygen_saturation.max"
MIN_COL = "com.samsung.health.oxygen_saturation.min"


def build_daily_summary(input_file: str = INPUT_FILE, output_file: str = OUTPUT_FILE) -> pd.DataFrame:
    if not input_file or not os.path.exists(input_file):
        raise FileNotFoundError(f"Input file not found or is invalid: {input_file}")
        
    # فایل خروجی Samsung Health یک سطر متادیتا قبل از هدر اصلی دارد.
    df = pd.read_csv(input_file, skiprows=1, dtype=str, low_memory=False)

    # تشخیص ستون زمان (برای پوشش حالت‌های احتمالی جابجایی ستون)
    time_candidates = [START_TIME_COL, CREATE_SH_VER_COL, PKG_NAME_COL]
    time_scores: dict[str, int] = {}
    parsed_times: dict[str, pd.Series] = {}

    for col in time_candidates:
        if col in df.columns:
            dt = pd.to_datetime(df[col], errors="coerce")
            parsed_times[col] = dt
            time_scores[col] = int(dt.notna().sum())

    if not time_scores:
        raise ValueError("No valid datetime column found for oxygen saturation dataset.")

    selected_time_col = max(time_scores, key=time_scores.get)
    if time_scores[selected_time_col] == 0:
        raise ValueError("Datetime parsing failed for oxygen saturation dataset.")

    # تشخیص ستون SpO2
    value_candidates = [SPO2_COL, MAX_COL, MIN_COL]
    value_scores: dict[str, int] = {}
    parsed_values: dict[str, pd.Series] = {}

    for col in value_candidates:
        if col in df.columns:
            numeric = pd.to_numeric(df[col], errors="coerce")
            parsed_values[col] = numeric
            value_scores[col] = int(numeric.notna().sum())

    if not value_scores:
        raise ValueError("No numeric oxygen saturation columns found.")

    selected_value_col = max(value_scores, key=value_scores.get)
    if value_scores[selected_value_col] == 0:
        raise ValueError("Could not parse numeric oxygen saturation values.")

    clean_df = pd.DataFrame(
        {
            "date": parsed_times[selected_time_col].dt.date,
            "spo2": parsed_values[selected_value_col],
        }
    )

    clean_df = clean_df.dropna(subset=["date", "spo2"])

    daily_summary = (
        clean_df.groupby("date", as_index=False)["spo2"]
        .agg(
            avg_spo2="mean",
            max_spo2="max",
            min_spo2="min",
        )
        .sort_values("date")
    )

    daily_summary["avg_spo2"] = daily_summary["avg_spo2"].round(2)

    daily_summary.to_csv(output_file, index=False, encoding="utf-8-sig")
    return daily_summary


if __name__ == "__main__":
    if INPUT_FILE:
        summary = build_daily_summary()
        print(summary.head(20).to_string(index=False))
        print(f"\nSaved: {OUTPUT_FILE}")
        print(f"Rows: {len(summary)}")
    else:
        print("Script did not run because the input file was not found.")


Found input file: New Data\com.samsung.shealth.tracker.oxygen_saturation.20260516153602.csv
      date  avg_spo2  max_spo2  min_spo2
2026-02-09     97.27      99.0      96.0
2026-02-10     97.64      99.0      96.0
2026-02-11     97.65      99.0      92.0
2026-02-12     97.62      99.0      96.0
2026-02-13     97.79      99.0      96.0
2026-02-14     97.81      99.0      96.0
2026-02-15     97.76      99.0      96.0
2026-02-16     97.56      99.0      92.0
2026-02-17     97.68      99.0      92.0
2026-02-18     97.33      99.0      92.0
2026-02-19     97.63      99.0      92.0
2026-02-20     97.73      99.0      92.0
2026-02-21     97.68      99.0      96.0
2026-02-22     97.49      99.0      96.0
2026-02-23     97.72      99.0      96.0
2026-02-24     97.64      99.0      92.0
2026-02-25     97.57      99.0      96.0
2026-02-26     96.83      98.0      93.0
2026-02-27     97.47      99.0      92.0
2026-02-28     97.48      99.0      92.0

Saved: daily_oxygen_saturation_summary.csv
Row